In [1]:
# Install required libraries
!pip install scikit-learn streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 71.9 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
# Load uploaded datasets
movies = pd.read_csv('/content/Movies.csv')
ratings = pd.read_csv('/content/Ratings.csv')

print("Movies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)

movies.head()

Movies Shape: (10329, 3)
Ratings Shape: (105339, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# Remove missing values
movies.dropna(inplace=True)
ratings.dropna(inplace=True)

# Merge datasets
data = pd.merge(ratings, movies, on='movieId')

data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,16,4.0,1217897793,Casino (1995),Crime|Drama
1,1,24,1.5,1217895807,Powder (1995),Drama|Sci-Fi
2,1,32,4.0,1217896246,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,4.0,1217896556,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,4.0,1217896523,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [5]:
user_movie_matrix = data.pivot_table(index='userId', columns='title', values='rating')

# Fill NaN with 0
user_movie_matrix.fillna(0, inplace=True)

user_movie_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...And Justice for All (1979),10 (1979),...,[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),a/k/a Tommy Chong (2005),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,659,660,661,662,663,664,665,666,667,668
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.101113,0.210044,0.128766,0.057896,0.077130,0.358090,0.097434,0.239189,0.026663,...,0.291162,0.144741,0.106583,0.091049,0.236805,0.154519,0.245071,0.238660,0.278217,0.153493
2,0.101113,1.000000,0.115559,0.034610,0.032705,0.028305,0.062914,0.471918,0.194232,0.000000,...,0.068325,0.000000,0.477330,0.146887,0.163553,0.061737,0.050948,0.051423,0.035907,0.064822
3,0.210044,0.115559,1.000000,0.058208,0.044426,0.012816,0.084522,0.066620,0.459703,0.068454,...,0.152078,0.301021,0.081626,0.098949,0.310234,0.079452,0.092821,0.080940,0.158943,0.109658
4,0.128766,0.034610,0.058208,1.000000,0.019298,0.005781,0.059089,0.024420,0.050572,0.000000,...,0.055860,0.024329,0.040467,0.108881,0.076241,0.014011,0.042643,0.174275,0.061677,0.157809
5,0.057896,0.032705,0.044426,0.019298,1.000000,0.053378,0.080822,0.041536,0.023168,0.011915,...,0.058450,0.007315,0.024708,0.038163,0.053085,0.048993,0.055431,0.026053,0.086667,0.068281


In [7]:
def recommend_collaborative(user_id, top_n=5):

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    similar_users_ratings = user_movie_matrix.loc[similar_users.index]

    weighted_ratings = similar_users_ratings.mean().sort_values(ascending=False)

    user_watched = user_movie_matrix.loc[user_id]
    unwatched_movies = user_watched[user_watched == 0]

    recommendations = weighted_ratings.loc[unwatched_movies.index]

    return recommendations.head(top_n)

In [8]:
tfidf = TfidfVectorizer(stop_words='english')

movies['genres'] = movies['genres'].str.replace('|', ' ')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

genre_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [9]:
def recommend_content(movie_title, top_n=5):

    idx = movies[movies['title'] == movie_title].index[0]

    similarity_scores = list(enumerate(genre_similarity[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:top_n+1]

    movie_indices = [i[0] for i in similarity_scores]

    return movies['title'].iloc[movie_indices]

In [10]:
def hybrid_recommend(user_id, movie_title, top_n=5):

    collab = recommend_collaborative(user_id, top_n)
    content = recommend_content(movie_title, top_n)

    hybrid_list = list(collab.index) + list(content)

    hybrid_unique = list(dict.fromkeys(hybrid_list))

    return hybrid_unique[:top_n]

In [11]:
print("Collaborative Recommendations:")
print(recommend_collaborative(user_id=1))

print("\nContent Based Recommendations:")
print(recommend_content(movie_title=movies['title'].iloc[0]))

print("\nHybrid Recommendations:")
print(hybrid_recommend(user_id=1, movie_title=movies['title'].iloc[0]))

Collaborative Recommendations:
title
'71 (2014)                                 0.0
'Hellboy': The Seeds of Creation (2004)    0.0
'Round Midnight (1986)                     0.0
'Til There Was You (1997)                  0.0
'burbs, The (1989)                         0.0
dtype: float64

Content Based Recommendations:
1815                                       Antz (1998)
2496                                Toy Story 2 (1999)
2967    Adventures of Rocky and Bullwinkle, The (2000)
3166                  Emperor's New Groove, The (2000)
3811                             Monsters, Inc. (2001)
Name: title, dtype: object

Hybrid Recommendations:
["'71 (2014)", "'Hellboy': The Seeds of Creation (2004)", "'Round Midnight (1986)", "'Til There Was You (1997)", "'burbs, The (1989)"]


In [12]:
from textblob import TextBlob

# Example: Assume you have a review column
# data['sentiment'] = data['review'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Filter positive sentiment movies
# positive_movies = data[data['sentiment'] > 0.2]

In [13]:
import streamlit as st
import pandas as pd

st.title("🎬 Movie Recommendation System")

user_id = st.number_input("Enter User ID", min_value=1, step=1)
movie_title = st.selectbox("Select Movie", movies['title'].values)

if st.button("Get Recommendations"):
    results = hybrid_recommend(user_id, movie_title)

    st.subheader("Top 5 Recommended Movies:")
    for movie in results:
        st.write(movie)

2026-02-26 11:07:32.278 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-26 11:07:32.516 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-02-26 11:07:32.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-26 11:07:32.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-26 11:07:32.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-26 11:07:32.522 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-26 11:07:32.524 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-26 11:07:32.526 Thread 'MainThread': mi

In [14]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py
